## Embdeddings and Vectorstores 

In [ ]:
import os
import openai
import sys
sys.path.append('../..')

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.environ['OPENAI_API_KEY']

In [2]:
from langchain_community.document_loaders import PyPDFLoader

# Load PDF
loaders = [
    # Duplicate documents on purpose - messy data
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture01.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture02.pdf"),
    PyPDFLoader("docs/MachineLearning-Lecture03.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())

In [3]:
# Split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1500,
    chunk_overlap = 150
)

In [4]:
splits = text_splitter.split_documents(docs)

In [5]:
len(splits)

208

## Embeddings

In [6]:
!pip install langchain_openai

In [7]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

In [8]:
sentence1 = "i like dogs"
sentence2 = "i like canines"
sentence3 = "the weather is ugly outside"

In [9]:
embedding1 = embedding.embed_query(sentence1)
embedding2 = embedding.embed_query(sentence2)
embedding3 = embedding.embed_query(sentence3)

In [10]:
embedding1

[-0.02755114622414112,
 -0.005382069852203131,
 -0.02582130767405033,
 -0.03305632621049881,
 -0.027349121868610382,
 0.02263941615819931,
 -0.01043584942817688,
 -0.008125188760459423,
 0.0025079497136175632,
 -0.01964692212641239,
 0.0006735478527843952,
 0.029243104159832,
 -0.005397852975875139,
 0.0007114275358617306,
 0.00021386229491326958,
 0.014078610576689243,
 0.02987443283200264,
 -0.001194787910208106,
 0.004210956394672394,
 -0.0038700394798070192,
 -0.01169219147413969,
 0.006875160150229931,
 0.012992726638913155,
 -0.04724857583642006,
 -0.002298033330589533,
 0.004706548992544413,
 0.016932211816310883,
 -0.0001317896822001785,
 -0.025884440168738365,
 -0.01635139063000679,
 0.02669254131615162,
 0.0029972288757562637,
 -0.015518037602305412,
 -0.02476067654788494,
 0.006035494152456522,
 -0.014949843287467957,
 0.00978558138012886,
 -0.01151541993021965,
 -0.004349848721176386,
 -0.010877778753638268,
 -0.0170584786683321,
 0.01118081621825695,
 0.006938292644917965,

In [11]:
import numpy as np

np.dot(embedding1, embedding2)

np.float64(0.9631511809630344)

In [12]:
np.dot(embedding1, embedding3)

np.float64(0.7702031371038217)

In [13]:
np.dot(embedding2, embedding3)

np.float64(0.7590540629791651)

In [14]:
! pip install chromadb

In [15]:
from langchain_community.vectorstores import Chroma

In [16]:
persist_directory = 'docs/chroma/'

In [17]:
!rm -rf ./docs/chroma  # remove old database files if any

In [18]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory=persist_directory
)

In [19]:
print(vectordb._collection.count())

208


## Similarity Search

In [20]:
question = "is there an email i can ask for help"

In [21]:
docs = vectordb.similarity_search(question,k=3)

In [22]:
len(docs)

3

In [23]:
docs[0].page_content

"cs229-qa@cs.stanford.edu. This goes to an account that's read by all the TAs and me. So \nrather than sending us email individually, if you send email to this account, it will \nactually let us get back to you maximally quickly with answers to your questions.  \nIf you're asking questions about homework problems, please say in the subject line which \nassignment and which question the email refers to, since that will also help us to route \nyour question to the appropriate TA or to me appropriately and get the response back to \nyou quickly.  \nLet's see. Skipping ahead — let's see — for homework, one midterm, one open and term \nproject. Notice on the honor code. So one thing that I think will help you to succeed and \ndo well in this class and even help you to enjoy this class more is if you form a study \ngroup.  \nSo start looking around where you're sitting now or at the end of class today, mingle a \nlittle bit and get to know your classmates. I strongly encourage you to form st

In [24]:
vectordb.persist()

/var/folders/1_/bwh3j9hx0fs5c8dhd9d9l_zc0000gn/T/ipykernel_47409/3711397106.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectordb.persist()


## Failure modes

In [25]:
question = "what did they say about matlab?"

In [26]:
docs = vectordb.similarity_search(question,k=5)

In [28]:
# Notice that we're getting duplicate chunks (because of the duplicate MachineLearning-Lecture01.pdf in the index).
# Semantic search fetches all similar documents, but does not enforce diversity docs[0] and docs[1] are indentical.

In [30]:
docs[0].page_content

'those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license, for the purposes of this class, there\'s also — [inaudible] \nwrite that down [inaudible] MATLAB — there\'s also a software package called Octave \nthat you can download for free off the Internet. And it has somewhat fewer features than \nMATLAB, but it\'s free, and for the purposes of this class, it will work for just abou

In [31]:
docs[1].page_content

'those homeworks will be done in either MATLAB or in Octave, which is sort of — I \nknow some people call it a free version of MATLAB, which it sort of is, sort of isn\'t.  \nSo I guess for those of you that haven\'t seen MATLAB before, and I know most of you \nhave, MATLAB is I guess part of the programming language that makes it very easy to \nwrite codes using matrices, to write code for numerical routines, to move data around, to \nplot data. And it\'s sort of an extremely easy to learn tool to use for implementing a lot of \nlearning algorithms.  \nAnd in case some of you want to work on your own home computer or something if you \ndon\'t have a MATLAB license, for the purposes of this class, there\'s also — [inaudible] \nwrite that down [inaudible] MATLAB — there\'s also a software package called Octave \nthat you can download for free off the Internet. And it has somewhat fewer features than \nMATLAB, but it\'s free, and for the purposes of this class, it will work for just abou

In [ ]:
# We can see a new failure mode.
# The question below asks a question about the third lecture, but includes results from other lectures as well.
